In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder\
                    .appName("aggreagtion")\
                    .master('local[*]')\
                    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/24 13:11:06 WARN Utils: Your hostname, Ameys-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 100.73.30.67 instead (on interface en0)
26/03/24 13:11:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/24 13:11:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [26]:
from pyspark.sql.functions import col, sum, avg, count, max, min,count,when,countDistinct


In [4]:
data = [
    (1, "Alice", "Laptop", 1200, "US"),
    (2, "Bob", "Phone", 800, "US"),
    (3, "Alice", "Mouse", 25, "UK"),
    (4, "David", "Laptop", 1300, "US"),
    (5, "Eve", "Keyboard", 75, "IN"),
    (6, "Frank", "Phone", 900, "US"),
    (7, "Alice", "Laptop", 1150, "UK"),
    (8, "Bob", "Monitor", 300, "US"),
    (9, "Alice", "Phone", 700, "UK")
]

columns = ["user_id", "user_name", "product", "amount", "country"]

orders_df = spark.createDataFrame(data, columns)

In [6]:
orders_df.groupBy(col('product'))\
         .agg(
             sum('amount').alias('total_amount'),
             avg('amount').alias('avg_amount'),
             count('*').alias('order_count')
         )

DataFrame[product: string, total_amount: bigint, avg_amount: double, count(1): bigint]

In [ ]:
orders_df.filter(col('country')=='US')\
    .groupBy('user_id')\
    .agg(
        sum('amount').alias('total_amount')
    )

In [9]:
orders_df.groupBy(col('country'))\
    .agg(
        count('product').alias('order_count'),
        sum('amount').alias('total_sales'),
        max('amount').alias('max_amount')
    )\
    .select(
        'country',
        'order_count',
        'total_sales',
        'max_amount'
    ).show()

[Stage 0:>                                                          (0 + 8) / 8]

+-------+-----------+-----------+----------+
|country|order_count|total_sales|max_amount|
+-------+-----------+-----------+----------+
|     US|          5|       4500|      1300|
|     UK|          3|       1875|      1150|
|     IN|          1|         75|        75|
+-------+-----------+-----------+----------+



In [ ]:
data = [
    (1, "Alice", "Laptop", 1200, "US"),
    (2, "Bob", "Phone", 800, "US"),
    (3, "Alice", "Mouse", 25, "UK"),
    (4, "David", "Laptop", 1300, "US"),
    (5, "Eve", "Keyboard", 75, "IN"),
    (6, "Frank", "Phone", 900, "US"),
    (7, "Alice", "Laptop", 1150, "UK"),
    (8, "Bob", "Monitor", 300, "US"),
    (9, "Alice", "Phone", 700, "UK")
]

columns = ["user_id", "user_name", "product", "amount", "country"]

In [12]:
orders_df.filter(col('amount')>=300)\
    .groupBy('user_name','country')\
    .agg(
        sum('amount').alias('total_amount'),
        count('product').alias('order_count')
    ).show()

+---------+-------+------------+-----------+
|user_name|country|total_amount|order_count|
+---------+-------+------------+-----------+
|    Alice|     US|        1200|          1|
|      Bob|     US|        1100|          2|
|    David|     US|        1300|          1|
|    Frank|     US|         900|          1|
|    Alice|     UK|        1850|          2|
+---------+-------+------------+-----------+



In [15]:
orders_df.groupBy(col('user_name'))\
    .agg(
        count(col('product')).alias('order_count')
    )\
    .filter(col('order_count')>1).show()

+---------+-----------+
|user_name|order_count|
+---------+-----------+
|    Alice|          4|
|      Bob|          2|
+---------+-----------+



In [24]:
df_with_colm = orders_df\
                .withColumn('us_amt', 
                            when(col('country')=='US',col('amount')).otherwise(0)
                )\
                .withColumn('non_us_amt', 
                            when(col('country')!='US',col('amount')).otherwise(0)
                )
df_with_colm.groupBy(col('user_name'))\
    .agg(
        sum('us_amt').alias('us_spend'),
        sum('non_us_amt').alias('non_us_spend')
    ).show()

+---------+--------+------------+
|user_name|us_spend|non_us_spend|
+---------+--------+------------+
|    Alice|    1200|        1875|
|      Bob|    1100|           0|
|    David|    1300|           0|
|      Eve|       0|          75|
|    Frank|     900|           0|
+---------+--------+------------+



In [27]:
orders_df.groupBy('country') \
    .agg(
        countDistinct('user_name').alias('distinct_users'),
        count('*').alias('total_orders')
    ) \
    .show()

+-------+--------------+------------+
|country|distinct_users|total_orders|
+-------+--------------+------------+
|     US|             4|           5|
|     IN|             1|           1|
|     UK|             1|           3|
+-------+--------------+------------+



In [28]:
# orders_fltrd = orders_df.withColumn(
#     'high_value',
#     when(col('amount')>=1000,1).otherwise(0)
#     .when(col('amount')<1000,1)
# )

orders_df.groupBy(col('country'))\
    .agg(
        sum(when(col('amount') >= 1000, 1).otherwise(0)).alias('high_value_order'),
        sum(when(col('amount') < 1000, 1).otherwise(0).alias('low_value_order'))
    ).show()

+-------+---------------------------------------------------------------------+-------------------------------------------------------------------+
|country|sum(CASE WHEN (amount >= 1000) THEN 1 ELSE 0 END AS high_value_order)|sum(CASE WHEN (amount < 1000) THEN 1 ELSE 0 END AS low_value_order)|
+-------+---------------------------------------------------------------------+-------------------------------------------------------------------+
|     US|                                                                    2|                                                                  3|
|     UK|                                                                    1|                                                                  2|
|     IN|                                                                    0|                                                                  1|
+-------+---------------------------------------------------------------------+---------------------------------